In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (stored for reuse)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")

# Convert timestamp to timezone-aware datetime in Asia/Shanghai and store as 'ts'
# Assume timestamps are epoch seconds in UTC
for df in (metric_df, trace_df, log_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure numeric 'value' columns
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Metrics aggregation: group by cmdb_id and kpi_name and compute stats
agg_funcs = {
    'count': ('value', 'count'),
    'min': ('value', 'min'),
    'max': ('value', 'max'),
    'mean': ('value', 'mean'),
}

metric_grouped = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(**agg_funcs).reset_index()

# compute percentiles separately to ensure correct ordering and names
quantiles = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile
p50 = quantiles(0.50).rename('P50').reset_index()
p90 = quantiles(0.90).rename('P90').reset_index()
p95 = quantiles(0.95).rename('P95').reset_index()
p99 = quantiles(0.99).rename('P99').reset_index()

# Merge percentiles into the aggregated df
metric_agg = metric_grouped.merge(p50, on=['cmdb_id', 'kpi_name'], how='left') \
                          .merge(p90, on=['cmdb_id', 'kpi_name'], how='left') \
                          .merge(p95, on=['cmdb_id', 'kpi_name'], how='left') \
                          .merge(p99, on=['cmdb_id', 'kpi_name'], how='left')

# Reorder columns as requested and sort by count desc, then take top 50
metric_agg = metric_agg[['cmdb_id', 'kpi_name', 'count', 'min', 'P50', 'P90', 'P95', 'P99', 'max', 'mean']]
metric_agg_top50 = metric_agg.sort_values(by='count', ascending=False).head(50)

# 2) Trace combinations with counts (top 50)
trace_counts = trace_df.groupby(['cmdb_id', 'trace_name']).size().reset_index(name='count')
trace_counts_top50 = trace_counts.sort_values('count', ascending=False).head(50)

# 3) Log combinations with counts (top 50)
log_counts = log_df.groupby(['cmdb_id', 'log_name']).size().reset_index(name='count')
log_counts_top50 = log_counts.sort_values('count', ascending=False).head(50)

# 4) Full set of distinct cmdb_id values appearing in any of the three files
cmdb_ids_metric = metric_df['cmdb_id'].dropna().unique()
cmdb_ids_trace = trace_df['cmdb_id'].dropna().unique()
cmdb_ids_log = log_df['cmdb_id'].dropna().unique()

distinct_cmdb_ids = pd.Series(sorted(set(cmdb_ids_metric).union(cmdb_ids_trace).union(cmdb_ids_log)), name='cmdb_id')

# Display compact outputs (variables only)
metric_agg_top50, trace_counts_top50, log_counts_top50, distinct_cmdb_ids

```
Out[1]:
```
```python
# Reuse previously computed variables from the kernel:
# metric_agg_top50, metric_agg, trace_counts_top50, trace_counts, log_counts_top50, log_counts, distinct_cmdb_ids

# Compact computed stats for the summary
total_metric_groups = int(metric_agg.shape[0])
metric_groups_count_31 = int((metric_agg['count'] == 31).sum())
metric_top3_examples = metric_agg_top50[['cmdb_id','kpi_name','count']].head(3).to_dict(orient='records')

total_trace_groups = int(trace_counts.shape[0])
trace_groups_count_30 = int((trace_counts['count'] == 30).sum())
trace_top3_examples = trace_counts_top50.head(3).to_dict(orient='records')

total_log_groups = int(log_counts.shape[0])
log_groups_count_31 = int((log_counts['count'] == 31).sum())
# highlight low-count log groups (count <= 1) for attention
low_count_logs = log_counts[log_counts['count'] <= 1].head(5).to_dict(orient='records')

distinct_cmdb_count = int(distinct_cmdb_ids.shape[0])
distinct_cmdb_sample = distinct_cmdb_ids.head(10).tolist()

summary = (
    f"Summary of telemetry aggregates (full-series analysis):\n\n"
    f"1) Metrics (metric.csv):\n"
    f"- Total distinct (cmdb_id, kpi_name) groups: {total_metric_groups}\n"
    f"- Groups with the maximum-observed count (31 samples): {metric_groups_count_31}\n"
    f"- Example top metric groups (cmdb_id, kpi_name, count): {metric_top3_examples}\n\n"
    f"2) Traces (trace.csv):\n"
    f"- Total distinct (cmdb_id, trace_name) groups: {total_trace_groups}\n"
    f"- Many trace groups have count = 30; number with count 30: {trace_groups_count_30}\n"
    f"- Example top trace groups (cmdb_id, trace_name, count): {trace_top3_examples}\n\n"
    f"3) Logs (log.csv):\n"
    f"- Total distinct (cmdb_id, log_name) groups: {total_log_groups}\n"
    f"- Groups with count = 31: {log_groups_count_31}\n"
    f"- Some log groups have very low counts (<=1); examples: {low_count_logs}\n\n"
    f"4) Distinct cmdb_id values across all files:\n"
    f"- Total distinct cmdb_id: {distinct_cmdb_count}\n"
    f"- Sample of cmdb_id values (up to 10 shown): {distinct_cmdb_sample}\n\n"
    f"Notes:\n"
    f"- The requested aggregations (count, min, P50, P90, P95, P99, max, mean) for metrics were computed\n"
    f"  and the top 50 groups by count are available in 'metric_agg_top50'.\n"
    f"- Trace and log group counts (top 50) are available in 'trace_counts_top50' and 'log_counts_top50'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                    kpi_name  count         min         P50         P90         P95         P99         max        mean
82             IG01  OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31    3.000000    3.000000    3.000000    3.000000    3.000000    3.000000    3.000000
1605       apache02                  OSLinux-CPU_CPU_CPUSysTime     31    0.195300    0.324163    0.457152    0.479966    0.516894    0.531849    0.331355
6              IG01    JVM-Threads_7778_JVM_ThreadCount_Threads     31   67.000000   70.000000   74.000000   74.000000   74.000000   74.000000   70.064516
1604       apache02                     OSLinux-CPU_CPU_CPULoad     31    0.000000    0.000000    0.040000    0.065000    0.094000    0.100000    0.014194
1608       apache02                 OSLinux-CPU_CPU_CPUidleutil     31   97.490027   99.459684   99.576183   99.615617   99.629400   99.634400   99.360341
...             ...                                         ...    ...         ...         ...         ...         ...         ...         ...         ...
1046  ServiceTest11                                     app.mrt     31    0.119840    0.334520    0.538880    0.703275    1.209498    1.401960    0.378672
1047  ServiceTest11                                      app.rr     31   98.250000  100.000000  100.000000  100.000000  100.000000  100.000000   99.943548
1049   ServiceTest2                                     app.cnt     31   23.000000   98.000000  166.000000  167.500000  177.800000  182.000000  102.645161
1050   ServiceTest2                                     app.mrt     31    0.180190    0.338810    0.650610    0.717505    0.792844    0.805630    0.371978
1051   ServiceTest2                                      app.rr     31  100.000000  100.000000  100.000000  100.000000  100.000000  100.000000  100.000000

[50 rows x 10 columns],    cmdb_id                         trace_name  count
0     IG01           trace.self.duration_mean     30
1     IG01            trace.self.duration_p95     30
2     IG01               trace.self.row_count     30
3     IG01    trace.to_Tomcat01.duration_mean     30
4     IG01     trace.to_Tomcat01.duration_p95     30
..     ...                                ...    ...
45    MG01  trace.from_dockerA2.duration_mean     30
46    MG01   trace.from_dockerA2.duration_p95     30
47    MG01      trace.from_dockerA2.row_count     30
48    MG01  trace.from_dockerB1.duration_mean     30
49    MG01   trace.from_dockerB1.duration_p95     30

[50 rows x 3 columns],      cmdb_id          log_name  count
9   Tomcat02     log.row_count     31
7   Tomcat02   log.error_count     31
6   Tomcat01     log.row_count     31
4   Tomcat01   log.error_count     31
18  apache02   log.error_count     31
17  apache01     log.row_count     31
19  apache02     log.row_count     31
16  apache01   log.error_count     31
10  Tomcat03   log.error_count     31
15  Tomcat04     log.row_count     31
13  Tomcat04   log.error_count     31
12  Tomcat03     log.row_count     31
8   Tomcat02  log.gc_pause_max     30
5   Tomcat01  log.gc_pause_max     16
11  Tomcat03  log.gc_pause_max     16
14  Tomcat04  log.gc_pause_max     16
1    Mysql01     log.row_count     14
0    Mysql01   log.error_count     14
3    Mysql02     log.row_count      1
2    Mysql02   log.error_count      1, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
28    dockerB2
Name: cmdb_id, Length: 29, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timezone Asia/Shanghai (UTC+8) and ts column exist
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if metric_df['ts'].dt.tz is None:
        metric_df['ts'] = pd.to_datetime(metric_df['ts']).dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        metric_df['ts'] = metric_df['ts'].dt.tz_convert(tz)

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P5 and P95 for each (cmdb_id, kpi_name)
p5 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'P5'})
p95 = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'P95'})
thresholds_df = p5.merge(p95, on=['cmdb_id','kpi_name'], how='outer')

# 2) Filter original metric rows to March 7, 2021, 17:30 to 18:00 (UTC+8) inclusive
start = pd.Timestamp('2021-03-07 17:30:00', tz=tz)
end = pd.Timestamp('2021-03-07 18:00:00', tz=tz)
metric_window_df = metric_df[(metric_df['ts'] >= start) & (metric_df['ts'] <= end)].copy()

# Merge thresholds into the windowed dataframe for comparisons
metric_window_df = metric_window_df.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left')

# 3) For each (cmdb_id, kpi_name) within the filtered window compute requested stats
def compute_window_stats(g):
    total_points = int(len(g))
    p95 = g['P95'].iloc[0]
    p5 = g['P5'].iloc[0]
    high_mask = g['value'] >= p95
    low_mask = g['value'] <= p5
    count_high = int(high_mask.sum())
    count_low = int(low_mask.sum())
    first_high_ts = g.loc[high_mask, 'ts'].min() if count_high > 0 else pd.NaT
    first_low_ts = g.loc[low_mask, 'ts'].min() if count_low > 0 else pd.NaT
    return pd.Series({
        'total_points_in_window': total_points,
        'P5': p5,
        'P95': p95,
        'count_high': count_high,
        'first_high_ts': first_high_ts,
        'count_low': count_low,
        'first_low_ts': first_low_ts
    })

window_stats = metric_window_df.groupby(['cmdb_id','kpi_name']).apply(compute_window_stats).reset_index()

# 4) Return top 20 groups sorted by (count_high + count_low) desc with requested columns
window_stats['extreme_count'] = window_stats['count_high'] + window_stats['count_low']
top20_results = window_stats.sort_values(by='extreme_count', ascending=False).head(20)[
    ['cmdb_id','kpi_name','total_points_in_window','P5','P95','count_high','first_high_ts','count_low','first_low_ts']
]

# Keep intermediate results in kernel variables: thresholds_df, metric_window_df, window_stats
thresholds_df, metric_window_df.shape, window_stats.shape, top20_results

```
Out[2]:
```
```python
# Summarize results from previous kernel computations in plain English (summary as a string)
n_thresholds = int(thresholds_df.shape[0])
window_rows = int(metric_window_df.shape[0])
window_groups = int(window_stats.shape[0])
groups_with_extremes = int((window_stats['count_high'] + window_stats['count_low'] > 0).sum())

top_examples = top20_results.head(5)[
    ['cmdb_id','kpi_name','total_points_in_window','P5','P95','count_high','count_low']
].copy()
# convert timestamps to ISO strings for compact display
for col in ('first_high_ts','first_low_ts'):
    if col in top_examples.columns:
        top_examples[col] = top_examples[col].astype(str)

summary = (
    f"Metric thresholds and windowed extremes summary:\n\n"
    f"- Global thresholds computed for {n_thresholds} distinct (cmdb_id, kpi_name) groups (P5 and P95).\n"
    f"- Incident window: 2021-03-07 17:30:00 to 2021-03-07 18:00:00 (UTC+8).\n"
    f"- Rows in window: {window_rows}. Distinct groups present in window: {window_groups}.\n"
    f"- Groups with at least one extreme point (<=P5 or >=P95) in the window: {groups_with_extremes}.\n\n"
    f"Top example groups by extreme occurrences (up to 5 shown):\n"
    f"{top_examples.to_string(index=False)}\n\n"
    f"Note: Many top groups show count_high and count_low equal to the full window length (31),\n"
    f"which indicates constant-valued series where P5 == P95 == observed value (all points meet both thresholds)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name            P5           P95
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
1         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.042800e+00  4.187080e+01
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  9.078029e+07  8.463759e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.812766e+07  5.812766e+07
4         IG01          JVM-Operating System_7778_JVM_JVM_CPULoad  2.985000e-01  3.934000e-01
...        ...                                                ...           ...           ...
1699  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  3.072000e+09  3.072000e+09
1700  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  9.416000e+01  9.416000e+01
1701  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  2.979544e+09  2.980193e+09
1702  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  1.803474e+06  3.734262e+06
1703  dockerB2  Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-...  1.818765e+05  2.189285e+05

[1704 rows x 4 columns], (27805, 7), (1704, 10),        cmdb_id                                           kpi_name  total_points_in_window            P5           P95  count_high             first_high_ts  count_low              first_low_ts
574    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1590  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31  4.000000e+00  4.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                      31  5.812766e+07  5.812766e+07          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                      31  2.040136e+09  2.040136e+09          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1504  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1515  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31  5.000000e+02  5.000000e+02          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1513  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31  1.000000e+02  1.000000e+02          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1396  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1397  Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1394  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1294  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
157       IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1295  Tomcat02  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1393  Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
158       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1398  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1399  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1395  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1403  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00
1402  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                      31  0.000000e+00  0.000000e+00          31 2021-03-07 17:30:00+08:00         31 2021-03-07 17:30:00+08:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables: window_stats, thresholds_df
# Candidate components list
candidates = [
    'apache01','apache02',
    'Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02',
    'IG01','IG02',
    'Mysql01','Mysql02',
    'Redis01','Redis02'
]

# Ensure window_stats exists
try:
    window_stats
except NameError:
    raise RuntimeError("window_stats not found in kernel. Run previous steps to produce window_stats.")

# Work on a copy scoped to candidate components
filtered_groups_candidates = window_stats[window_stats['cmdb_id'].isin(candidates)].copy()

# Ensure numeric counts
filtered_groups_candidates['count_high'] = filtered_groups_candidates['count_high'].fillna(0).astype(int)
filtered_groups_candidates['count_low'] = filtered_groups_candidates['count_low'].fillna(0).astype(int)

# Compute per-row helpers
filtered_groups_candidates['extreme_count'] = filtered_groups_candidates['count_high'] + filtered_groups_candidates['count_low']

# Ensure P5 and P95 are present and numeric
filtered_groups_candidates['P5'] = pd.to_numeric(filtered_groups_candidates['P5'], errors='coerce')
filtered_groups_candidates['P95'] = pd.to_numeric(filtered_groups_candidates['P95'], errors='coerce')

# faulty_flag: extreme > 0 AND P95 != P5 AND both thresholds not null (exclude trivial constant-series)
filtered_groups_candidates['faulty_flag'] = (
    (filtered_groups_candidates['extreme_count'] > 0) &
    (~filtered_groups_candidates['P5'].isna()) &
    (~filtered_groups_candidates['P95'].isna()) &
    (filtered_groups_candidates['P95'] != filtered_groups_candidates['P5'])
)

# per-group earliest fault timestamp (min of first_high_ts and first_low_ts)
# Ensure ts columns are datetimes (they should be), otherwise coerce
for col in ['first_high_ts','first_low_ts']:
    if col in filtered_groups_candidates.columns:
        filtered_groups_candidates[col] = pd.to_datetime(filtered_groups_candidates[col])

filtered_groups_candidates['group_first_fault_ts'] = filtered_groups_candidates[['first_high_ts','first_low_ts']].min(axis=1)

# Aggregations per component
def compile_component_group(g):
    num_groups_in_window = int(g.shape[0])
    num_faulty_groups = int(g['faulty_flag'].sum())
    # earliest_fault_ts among faulty groups
    faulty_rows = g[g['faulty_flag']]
    if not faulty_rows.empty:
        earliest_fault_ts = faulty_rows['group_first_fault_ts'].min()
    else:
        earliest_fault_ts = pd.NaT
    total_high = int(g['count_high'].sum())
    total_low = int(g['count_low'].sum())
    # sample faulty kpis up to 3 prioritized by extreme_count desc
    if not faulty_rows.empty:
        sample_kpis = faulty_rows.sort_values('extreme_count', ascending=False)['kpi_name'].dropna().unique()[:3].tolist()
    else:
        sample_kpis = []
    return pd.Series({
        'num_groups_in_window': num_groups_in_window,
        'num_faulty_groups': num_faulty_groups,
        'earliest_fault_ts': earliest_fault_ts,
        'total_high': total_high,
        'total_low': total_low,
        'sample_faulty_kpis': ', '.join(sample_kpis)
    })

component_summary = filtered_groups_candidates.groupby('cmdb_id').apply(compile_component_group).reset_index().rename(columns={'cmdb_id':'component'})

# Sort by num_faulty_groups desc and limit to top 20 (there are <=14 candidates)
component_summary_top20 = component_summary.sort_values(by='num_faulty_groups', ascending=False).head(20)

# Display only the compact per-component table (max 20 rows)
component_summary_top20

```
Out[3]:
```
```python
# Summarize the component-level fault table (component_summary_top20) in plain English.
# Reuse kernel variable component_summary_top20 produced earlier.

rows = []
for r in component_summary_top20.itertuples(index=False):
    comp = r.component
    num_groups = int(r.num_groups_in_window)
    num_faulty = int(r.num_faulty_groups)
    ets = r.earliest_fault_ts
    ets_str = ets.isoformat() if pd.notna(ets) else "None"
    total_high = int(r.total_high)
    total_low = int(r.total_low)
    sample_kpis = r.sample_faulty_kpis if r.sample_faulty_kpis else "(none)"
    rows.append(f"- {comp}: {num_faulty} faulty groups / {num_groups} groups; earliest_fault_ts={ets_str}; total_high={total_high}; total_low={total_low}; sample_kpis={sample_kpis}")

summary = (
    "Per-component fault summary for window 2021-03-07 17:30–18:00 (UTC+8):\n\n"
    + "\n".join(rows)
    + "\n\nNotes: Components are sorted by num_faulty_groups. "
    "Mysql02 and Mysql01 show the most faulty groups (116 and 105 respectively). "
    "Many components have earliest faults at 2021-03-07T17:30:00+08:00, and sample_kpis lists up to three example KPI names per component."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component  num_groups_in_window  num_faulty_groups         earliest_fault_ts  total_high  total_low                                 sample_faulty_kpis
5    Mysql02                   226                116 2021-03-07 17:30:00+08:00        2204       2401  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
4    Mysql01                   225                105 2021-03-07 17:30:00+08:00        2372       2525  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...
0       IG01                    90                 54 2021-03-07 17:30:00+08:00         650        708  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...
1       IG02                    90                 53 2021-03-07 17:30:00+08:00         621        722  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
12  apache01                    79                 52 2021-03-07 17:30:00+08:00         344        336  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
8   Tomcat01                   108                 52 2021-03-07 17:30:00+08:00         897        967  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...
10  Tomcat03                   107                 51 2021-03-07 17:30:00+08:00        1039       1070  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc, ...
6    Redis01                   113                 51 2021-03-07 17:30:00+08:00        1347       1354  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
9   Tomcat02                   111                 50 2021-03-07 17:30:00+08:00         982       1040  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
11  Tomcat04                   109                 49 2021-03-07 17:30:00+08:00        1045       1074  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
13  apache02                    81                 49 2021-03-07 17:30:00+08:00         598        655  OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCou...
7    Redis02                   109                 47 2021-03-07 17:30:00+08:00        1368       1405  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
2       MG01                    92                 44 2021-03-07 17:30:00+08:00         655        745  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem, OSLinu...
3       MG02                    92                 38 2021-03-07 17:30:00+08:00         693        753  JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed, O...```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timezone Asia/Shanghai (UTC+8) and ts column exist
tz = pytz.timezone('Asia/Shanghai')
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if trace_df['ts'].dt.tz is None:
        trace_df['ts'] = pd.to_datetime(trace_df['ts']).dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        trace_df['ts'] = trace_df['ts'].dt.tz_convert(tz)

# Ensure numeric 'value'
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Restrict to candidate components for analysis
candidates = ['Mysql02','Mysql01','IG01','IG02','apache01','Tomcat01']
trace_candidates_full = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# 2) Compute global P5 and P95 per (cmdb_id, trace_name) using full-series
p5_trace = trace_candidates_full.groupby(['cmdb_id','trace_name'])['value'].quantile(0.05).reset_index().rename(columns={'value':'P5'})
p95_trace = trace_candidates_full.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'P95'})
thresholds_trace_df = p5_trace.merge(p95_trace, on=['cmdb_id','trace_name'], how='outer')

# 3) Filter trace rows to incident window 2021-03-07 17:30 to 18:00 (UTC+8)
start = pd.Timestamp('2021-03-07 17:30:00', tz=tz)
end = pd.Timestamp('2021-03-07 18:00:00', tz=tz)
trace_window_df = trace_candidates_full[(trace_candidates_full['ts'] >= start) & (trace_candidates_full['ts'] <= end)].copy()

# Merge thresholds into the windowed dataframe for comparisons
trace_window_df = trace_window_df.merge(thresholds_trace_df, on=['cmdb_id','trace_name'], how='left')

# 4) For each (cmdb_id, trace_name) compute requested stats
def compute_trace_window_stats(g):
    total_points = int(len(g))
    p95 = g['P95'].iloc[0]
    p5 = g['P5'].iloc[0]
    high_mask = g['value'] >= p95
    low_mask = g['value'] <= p5
    count_high = int(high_mask.sum())
    count_low = int(low_mask.sum())
    first_high_ts = g.loc[high_mask, 'ts'].min() if count_high > 0 else pd.NaT
    first_low_ts = g.loc[low_mask, 'ts'].min() if count_low > 0 else pd.NaT
    return pd.Series({
        'total_points_in_window': total_points,
        'P5': p5,
        'P95': p95,
        'count_high': count_high,
        'first_high_ts': first_high_ts,
        'count_low': count_low,
        'first_low_ts': first_low_ts
    })

trace_window_stats = trace_window_df.groupby(['cmdb_id','trace_name']).apply(compute_trace_window_stats).reset_index()

# Prepare final compact table sorted by (count_high + count_low) desc, limit 20 rows
trace_window_stats['extreme_count'] = trace_window_stats['count_high'] + trace_window_stats['count_low']
trace_top20_results = trace_window_stats.sort_values(by='extreme_count', ascending=False).head(20)[
    ['cmdb_id','trace_name','total_points_in_window','P5','P95','count_high','first_high_ts','count_low','first_low_ts']
].reset_index(drop=True)

# Keep intermediate variables: thresholds_trace_df, trace_window_df, trace_window_stats
thresholds_trace_df.shape, trace_window_df.shape, trace_window_stats.shape, trace_top20_results

```
Out[4]:
```
```python
# Summarize trace analysis results in plain English using kernel variables
n_threshold_groups = int(thresholds_trace_df.shape[0])  # 48
rows_in_window = int(trace_window_df.shape[0])  # 1353
n_window_groups = int(trace_window_stats.shape[0])  # 48

# Top row example
top = trace_top20_results.iloc[0]
top_example = (
    f"{top['cmdb_id']} / {top['trace_name']}: total_points={int(top['total_points_in_window'])}, "
    f"P5={float(top['P5'])}, P95={float(top['P95'])}, "
    f"count_high={int(top['count_high'])} (first {top['first_high_ts']}), "
    f"count_low={int(top['count_low'])} (first {top['first_low_ts']})"
)

summary = (
    "Trace analysis summary (full-series thresholds; window 2021-03-07 17:30–18:00 UTC+8):\n\n"
    f"- Candidate components analyzed: Mysql02, Mysql01, IG01, IG02, apache01, Tomcat01.\n"
    f"- Global thresholds (P5/P95) were computed for {n_threshold_groups} distinct (cmdb_id, trace_name) groups.\n"
    f"- Windowed data: {rows_in_window} rows across {n_window_groups} (cmdb_id, trace_name) groups.\n\n"
    f"- Top example group by extreme occurrences:\n  {top_example}\n\n"
    "Key findings:\n"
    "- Many trace groups (especially for IG01 and IG02) show both high and low extremes in the window; typical pattern is\n"
    "  count_high = 2 and count_low = 2 for several trace metrics (e.g., duration_mean, row_count, duration_p95).\n"
    "- Earliest extreme timestamps occur early in the window (some first highs at ~2021-03-07 17:30:00+08:00) and many lows\n"
    "  around ~17:42:00+08:00; some high-duration spikes are later (~17:54+08:00).\n"
    "- Overall the IG nodes show the most trace activity with extremes targeting various Tomcat endpoints (to_TomcatXX).\n\n"
    "If you want the detailed top-20 table returned earlier, inspect 'trace_top20_results' in the kernel."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((48, 4), (1353, 7), (48, 10),    cmdb_id                       trace_name  total_points_in_window           P5          P95  count_high             first_high_ts  count_low              first_low_ts
0     IG02      trace.to_Tomcat04.row_count                      30   132.000000   180.600000           2 2021-03-07 17:30:00+08:00          3 2021-03-07 17:42:00+08:00
1     IG01         trace.self.duration_mean                      30     0.241323     0.855407           2 2021-03-07 17:55:00+08:00          2 2021-03-07 17:42:00+08:00
2     IG01             trace.self.row_count                      30  1014.000000  1453.800000           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:37:00+08:00
3     IG01          trace.self.duration_p95                      30     1.077147     3.900050           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:42:00+08:00
4     IG01   trace.to_Tomcat01.duration_p95                      30     1.066615     5.619840           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:40:00+08:00
5     IG01      trace.to_Tomcat01.row_count                      30   117.850000   189.600000           2 2021-03-07 17:32:00+08:00          2 2021-03-07 17:37:00+08:00
6     IG01  trace.to_Tomcat02.duration_mean                      30     0.200728     0.680129           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:42:00+08:00
7     IG01  trace.to_Tomcat01.duration_mean                      30     0.217482     2.494228           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:40:00+08:00
8     IG01      trace.to_Tomcat02.row_count                      30   125.700000   177.100000           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:35:00+08:00
9     IG01  trace.to_Tomcat03.duration_mean                      30     0.194007     0.634351           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:42:00+08:00
10    IG01   trace.to_Tomcat03.duration_p95                      30     1.059443     3.044730           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:52:00+08:00
11    IG01      trace.to_Tomcat03.row_count                      30   119.900000   185.650000           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:37:00+08:00
12    IG01  trace.to_Tomcat04.duration_mean                      30     0.213378     0.626505           2 2021-03-07 17:31:00+08:00          2 2021-03-07 17:43:00+08:00
13    IG01   trace.to_Tomcat04.duration_p95                      30     1.053210     2.810352           2 2021-03-07 17:31:00+08:00          2 2021-03-07 17:42:00+08:00
14    IG01      trace.to_Tomcat04.row_count                      30   120.700000   187.150000           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:37:00+08:00
15    IG01   trace.to_Tomcat02.duration_p95                      30     1.059697     3.141960           2 2021-03-07 17:31:00+08:00          2 2021-03-07 17:42:00+08:00
16    IG02         trace.self.duration_mean                      30     0.234599     0.755998           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:42:00+08:00
17    IG02          trace.self.duration_p95                      30     1.081575     3.328200           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:42:00+08:00
18    IG02  trace.to_Tomcat01.duration_mean                      30     0.170957     2.036909           2 2021-03-07 17:54:00+08:00          2 2021-03-07 17:40:00+08:00
19    IG02             trace.self.row_count                      30  1044.900000  1454.200000           2 2021-03-07 17:30:00+08:00          2 2021-03-07 17:42:00+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing log_df if present; otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Load error_logs.csv (store in kernel variable)
try:
    error_logs_df
except NameError:
    error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timezone Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Prepare ts columns for both dataframes if not present
for df in (log_df, error_logs_df):
    if 'ts' not in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        # ensure tz-aware and converted
        if df['ts'].dt.tz is None:
            df['ts'] = pd.to_datetime(df['ts']).dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            df['ts'] = pd.to_datetime(df['ts']).dt.tz_convert(tz)

# Ensure numeric 'value' in log_df
if 'value' in log_df.columns:
    log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
else:
    log_df['value'] = pd.NA

# Incident window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-07 17:30:00', tz=tz)
end = pd.Timestamp('2021-03-07 18:00:00', tz=tz)

# Candidate components
candidates = ['Mysql01','Mysql02','IG01','Tomcat01']

# 1) error_logs.csv processing: filter and aggregate
error_window_df = error_logs_df[
    (error_logs_df['cmdb_id'].isin(candidates)) &
    (error_logs_df['ts'] >= start) & (error_logs_df['ts'] <= end)
].copy()

# Aggregate per component: total rows, earliest ts, up to 5 distinct sample messages
def sample_messages(series):
    # take up to 5 distinct non-null messages
    uniques = pd.Series(series.dropna().unique())
    sampled = uniques.head(5).astype(str).tolist()
    return "; ".join(sampled)

error_summary = (
    error_window_df
    .groupby('cmdb_id')
    .agg(
        total_error_rows_in_window = ('message', 'count'),
        earliest_error_ts_in_window = ('ts', 'min'),
        sample_messages = ('message', sample_messages)
    )
    .reset_index()
    .sort_values(by='total_error_rows_in_window', ascending=False)
)

# Limit to at most 4 rows (one per component)
error_summary_top4 = error_summary.head(4)

# 2) log.csv processing: filter and aggregate per (component, log_name)
log_window_df = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['ts'] >= start) & (log_df['ts'] <= end)
].copy()

# For earliest_ts_where_value>0_in_window compute min ts where value>0 per group
def earliest_positive_ts(sub):
    pos = sub.loc[sub['value'] > 0, 'ts']
    return pos.min() if not pos.empty else pd.NaT

log_summary = (
    log_window_df
    .groupby(['cmdb_id', 'log_name'])
    .agg(
        total_points_in_window = ('value', 'count'),
        sum_value_in_window = ('value', 'sum'),
        max_value_in_window = ('value', 'max')
    )
    .reset_index()
)

# compute earliest positive ts separately and merge
earliest_pos = (
    log_window_df.groupby(['cmdb_id', 'log_name'])
    .apply(earliest_positive_ts)
    .reset_index(name='earliest_ts_where_value_gt0_in_window')
)

log_summary = log_summary.merge(earliest_pos, on=['cmdb_id','log_name'], how='left')

# Sort by total_points_in_window desc and take top 20
log_summary_top20 = log_summary.sort_values(by='total_points_in_window', ascending=False).head(20)

# Keep full filtered tables in kernel variables: error_window_df, log_window_df
# Display only the compact aggregate tables as requested
error_summary_top4, log_summary_top20

```
Out[5]:
```
```python
# Summarize error_logs and log aggregates (results from previous kernel variables)
try:
    error_summary_top4
    log_summary_top20
except NameError:
    raise RuntimeError("Required kernel variables not found: error_summary_top4, log_summary_top20")

def ts_to_str(t):
    return t.isoformat() if pd.notna(t) else "None"

# Error logs summary
if error_summary_top4.empty:
    error_part = "Error logs: No error_log rows for Mysql01, Mysql02, IG01, or Tomcat01 in the incident window."
else:
    rows = []
    for r in error_summary_top4.itertuples(index=False):
        comp = r.cmdb_id
        total = int(r.total_error_rows_in_window)
        ets = ts_to_str(r.earliest_error_ts_in_window)
        samples = r.sample_messages if r.sample_messages else "(no sample messages)"
        rows.append(f"{comp}: total_error_rows={total}; earliest={ets}; samples={samples}")
    error_part = "Error logs:\n" + "\n".join(rows)

# Log summary per (component, log_name)
# Group rows by component for clearer presentation
log_parts = []
if log_summary_top20.empty:
    log_part = "Logs: No log rows for the specified components in the incident window."
else:
    grouped = {}
    for r in log_summary_top20.itertuples(index=False):
        comp = r.cmdb_id
        grouped.setdefault(comp, []).append(r)
    for comp in sorted(grouped.keys()):
        entries = grouped[comp]
        lines = [f"- {comp}:"]
        for e in entries:
            ln = e.log_name
            total = int(e.total_points_in_window)
            ssum = float(e.sum_value_in_window) if not pd.isna(e.sum_value_in_window) else 0.0
            mmax = float(e.max_value_in_window) if not pd.isna(e.max_value_in_window) else 0.0
            first_pos = ts_to_str(e.earliest_ts_where_value_gt0_in_window)
            lines.append(f"    • {ln}: points={total}, sum={ssum:.2f}, max={mmax:.2f}, first_value>0={first_pos}")
        log_parts.append("\n".join(lines))
    log_part = "Logs (per component and log_name):\n" + "\n".join(log_parts)

summary = (
    "Incident window (2021-03-07 17:30–18:00 UTC+8) — error_logs and log.csv summary:\n\n"
    f"{error_part}\n\n"
    f"{log_part}\n\n"
    "Key highlights:\n"
    "- No error_log rows found for the four components in the window.\n"
    "- Tomcat01 has the most log activity: log.row_count (31 pts, sum=9273, max=377, first>0 at 2021-03-07T17:30:00+08:00),\n"
    "  log.error_count present but all zeros, and log.gc_pause_max shows non-zero pauses (first>0 at ~17:31).\n"
    "- Mysql01 and Mysql02 have log.row_count records (Mysql01: 14 pts, sum=35, first>0 at ~17:31; Mysql02: 1 pt at ~17:55).\n"
    "- Where log.error_count exists for Mysql and Tomcat it sums to 0 in this window (no positive error counts observed).\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, total_error_rows_in_window, earliest_error_ts_in_window, sample_messages]
Index: [],     cmdb_id          log_name  total_points_in_window  sum_value_in_window  max_value_in_window earliest_ts_where_value_gt0_in_window
4  Tomcat01   log.error_count                      31                 0.00                 0.00                                   NaT
6  Tomcat01     log.row_count                      31              9273.00               377.00             2021-03-07 17:30:00+08:00
5  Tomcat01  log.gc_pause_max                      16                 1.13                 0.14             2021-03-07 17:31:00+08:00
0   Mysql01   log.error_count                      14                 0.00                 0.00                                   NaT
1   Mysql01     log.row_count                      14                35.00                 7.00             2021-03-07 17:31:00+08:00
2   Mysql02   log.error_count                       1                 0.00                 0.00                                   NaT
3   Mysql02     log.row_count                       1                 1.00                 1.00             2021-03-07 17:55:00+08:00)```
```